# CIFAR-10 Neural Network Assignment

This notebook trains different neural network models on CIFAR-10 and analyzes how depth, learning rate, and batch size affect performance.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import glob

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BATCH_SIZE = 64
LEARNING_RATE = 0.1
EPOCHS = 20
SUBSET_SIZE = 1.0

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/gdrive')
    SAVE_PATH = '/content/gdrive/My Drive/CIFAR10_Results/'
    print("Running on Google Colab")
except:
    IN_COLAB = False
    SAVE_PATH = './'
    print("Running locally")

os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
def load_data_cifar10(subset_size=1.0):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )
    
    train_size = int(len(trainset) * subset_size)
    test_size = int(len(testset) * subset_size)
    
    trainset, _ = random_split(
        trainset, [train_size, len(trainset) - train_size],
        generator=torch.Generator().manual_seed(0)
    )
    testset, _ = random_split(
        testset, [test_size, len(testset) - test_size],
        generator=torch.Generator().manual_seed(0)
    )
    
    val_size = int(len(trainset) * 0.2)
    train_ds, val_ds = random_split(
        trainset, [len(trainset) - val_size, val_size],
        generator=torch.Generator().manual_seed(0)
    )
    
    print(f"Training: {len(train_ds)}, Validation: {len(val_ds)}, Test: {len(testset)}")
    return train_ds, val_ds, testset

def load_batches(train_ds, val_ds, test_ds, batch_size=128):
    train_iter = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_iter = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_iter = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_iter, val_iter, test_iter

train_ds, val_ds, test_ds = load_data_cifar10(SUBSET_SIZE)
train_iter, val_iter, test_iter = load_batches(train_ds, val_ds, test_ds, BATCH_SIZE)

## Models

In [ ]:
class Softmax(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3 * 32 * 32, 10)
    
    def forward(self, x):
        return self.fc(x.view(x.size(0), -1))

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        return self.fc(x.view(x.size(0), -1))

def create_cnn(depth=2):
    layers = []
    in_ch, out_ch = 3, 32
    pool_count = 0
    max_pools = 3
    
    for i in range(depth):
        layers.append(nn.Conv2d(in_ch, out_ch, 3, padding=1))
        layers.append(nn.ReLU())
        
        should_pool = (i + 1) % 2 == 0 and pool_count < max_pools
        if should_pool:
            layers.append(nn.MaxPool2d(2, 2))
            pool_count += 1
        
        in_ch = out_ch
        if i % 4 == 1:
            out_ch = min(out_ch * 2, 256)
    
    layers.append(nn.AdaptiveAvgPool2d(1))
    return nn.Sequential(*layers)

class CNN(nn.Module):
    def __init__(self, depth=2):
        super().__init__()
        self.conv = create_cnn(depth).to(device)
        
        with torch.no_grad():
            x = torch.randn(1, 3, 32, 32).to(device)
            x = self.conv(x)
            flat_size = x.view(1, -1).size(1)
        
        self.fc = nn.Sequential(
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

## Training Functions

In [ ]:
def accuracy(y_hat, y):
    return float((torch.argmax(y_hat, dim=1) == y).sum()) / len(y)

def train_epoch(net, train_iter, loss_fn, optimizer):
    net.train()
    total_loss = 0
    num_batches = 0
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)
        y_hat = net(X)
        loss = loss_fn(y_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / num_batches

def evaluate(net, data_iter, loss_fn):
    net.eval()
    total_loss = 0
    total_acc = 0
    num_batches = 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            loss = loss_fn(y_hat, y)
            total_loss += loss.item()
            total_acc += accuracy(y_hat, y)
            num_batches += 1
    return total_loss / num_batches, total_acc / num_batches

def train(net, train_iter, val_iter, test_iter, epochs, lr, weight_decay=0):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, weight_decay=weight_decay)
    train_losses, val_losses, val_accs = [], [], []
    start_time = time.time()
    
    for epoch in range(epochs):
        train_loss = train_epoch(net, train_iter, loss_fn, optimizer)
        val_loss, val_acc = evaluate(net, val_iter, loss_fn)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs}: Loss={train_loss:.3f}, Val={val_loss:.3f}, Acc={val_acc:.3f}")
    
    elapsed = time.time() - start_time
    test_loss, test_acc = evaluate(net, test_iter, loss_fn)
    print(f"Test Accuracy: {test_acc:.4f}, Time: {elapsed:.0f}s\n")
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'test_acc': test_acc,
        'train_time': elapsed
    }

## Experiment 1: Softmax Training

In [ ]:
print("\nPART 1: SOFTMAX TRAINING")
print("="*60)

softmax_net = Softmax().to(device)
softmax_result = train(softmax_net, train_iter, val_iter, test_iter, EPOCHS, LEARNING_RATE)

print("[Results]")
print(f"Test Accuracy: {softmax_result['test_acc']:.4f}")
print(f"\n[Errors by Epoch]")
for epoch in range(len(softmax_result['train_losses'])):
    print(f"Epoch {epoch+1}: Train={softmax_result['train_losses'][epoch]:.4f}, Val={softmax_result['val_losses'][epoch]:.4f}, Acc={softmax_result['val_accs'][epoch]:.4f}")

## Experiment 2A: Depth Analysis

In [ ]:
print("\nPART 2A: DEPTH ANALYSIS")
print("="*60)

depths = [2, 8, 16, 32]
results_depth = {}

for depth in depths:
    print(f"Training depth {depth}...")
    net = CNN(depth=depth).to(device)
    result = train(net, train_iter, val_iter, test_iter, EPOCHS, LEARNING_RATE)
    results_depth[depth] = result

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
for d in depths:
    ax.plot(results_depth[d]['train_losses'], label=f'Depth {d}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for d in depths:
    ax.plot(results_depth[d]['val_losses'], label=f'Depth {d}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for d in depths:
    ax.plot(results_depth[d]['val_accs'], label=f'Depth {d}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
test_accs = [results_depth[d]['test_acc'] for d in depths]
ax.bar([str(d) for d in depths], test_accs, color='steelblue', alpha=0.7)
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Depth')
ax.set_title('Final Test Accuracy')
ax.grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(test_accs):
    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'experiment_a_depth.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: experiment_a_depth.png")

## Experiment 2B: Learning Rate Analysis

In [ ]:
print("\nPART 2B: LEARNING RATE ANALYSIS")
print("="*60)

learning_rates = [0.000001, 0.0001, 0.001, 0.01, 1.0]
results_lr = {}

for lr in learning_rates:
    print(f"Training LR={lr}...")
    net = SimpleCNN().to(device)
    result = train(net, train_iter, val_iter, test_iter, EPOCHS, lr)
    results_lr[lr] = result

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
for lr in learning_rates:
    ax.plot(results_lr[lr]['train_losses'], label=f'LR={lr}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for lr in learning_rates:
    ax.plot(results_lr[lr]['val_losses'], label=f'LR={lr}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for lr in learning_rates:
    ax.plot(results_lr[lr]['val_accs'], label=f'LR={lr}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
test_accs = [results_lr[lr]['test_acc'] for lr in learning_rates]
ax.bar([f'{lr:.6f}' for lr in learning_rates], test_accs, color='coral', alpha=0.7)
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Learning Rate')
ax.set_title('Final Test Accuracy')
ax.grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(test_accs):
    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'experiment_b_lr.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: experiment_b_lr.png")

## Experiment 2C: Batch Size Analysis

In [ ]:
print("\nPART 2C: BATCH SIZE ANALYSIS")
print("="*60)

batch_sizes = [1, 8, 16, 64, 256]
results_bs = {}

for bs in batch_sizes:
    print(f"Training batch size {bs}...")
    train_iter_bs, val_iter_bs, test_iter_bs = load_batches(train_ds, val_ds, test_ds, batch_size=bs)
    net = SimpleCNN().to(device)
    result = train(net, train_iter_bs, val_iter_bs, test_iter_bs, EPOCHS, LEARNING_RATE)
    results_bs[bs] = result

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
for bs in batch_sizes:
    ax.plot(results_bs[bs]['train_losses'], label=f'BS={bs}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for bs in batch_sizes:
    ax.plot(results_bs[bs]['val_losses'], label=f'BS={bs}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for bs in batch_sizes:
    ax.plot(results_bs[bs]['val_accs'], label=f'BS={bs}', marker='o')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
test_accs = [results_bs[bs]['test_acc'] for bs in batch_sizes]
ax.bar([str(bs) for bs in batch_sizes], test_accs, color='lightgreen', alpha=0.7)
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Batch Size')
ax.set_title('Final Test Accuracy')
ax.grid(True, axis='y', alpha=0.3)
for i, acc in enumerate(test_accs):
    ax.text(i, acc + 0.01, f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'experiment_c_bs.png'), dpi=100, bbox_inches='tight')
plt.show()
print("Saved: experiment_c_bs.png")

In [ ]:
saved_files = glob.glob(os.path.join(SAVE_PATH, '*.png'))
for f in sorted(saved_files):
    print(f"  {os.path.basename(f)}")

print(f"\nLocation: {SAVE_PATH}")

if IN_COLAB:
    print("  Files > My Drive > CIFAR10_Results/")
else:
    print("\nFiles saved locally")